# Prompt Values API Reference


# `PromptValue: Serializable, ABC`

Abstract base class for values that can be converted to both text-generation inputs and chat-model messages.

Concrete subclasses must implement `to_string` and `to_messages`. The module provides no asynchronous wrappers or optional subclass hooks. The abstract methods do not explicitly raise `NotImplementedError`.

## Methods

### `is_lc_serializable`

Returns `True`, marking prompt values as LangChain-serializable.

```python
@classmethod
is_lc_serializable(
    cls,
) -> bool # Always True
```

### `get_lc_namespace`

Returns the default serialization namespace.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # ["langchain", "schema", "prompt"]
```

## Required subclass hooks

### `to_string`

Converts the prompt value to text.

```python
@abstractmethod
to_string(
    self,
) -> str # String representation
```

### `to_messages`

Converts the prompt value to chat messages.

```python
@abstractmethod
to_messages(
    self,
) -> list[BaseMessage] # Message representation
```

In [ ]:
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage # Import LangChain message types
from langchain_core.prompt_values import PromptValue # Import the abstract prompt-value class


class InstructionPromptValue(PromptValue): # Define a concrete prompt-value implementation
    instruction: str # Store the system instruction
    question: str # Store the user's question

    def to_string(self) -> str: # Convert the prompt into a single text string
        return f"Instruction: {self.instruction}\nQuestion: {self.question}" # Combine both values into one prompt

    def to_messages(self) -> list[BaseMessage]: # Convert the prompt into chat-model messages
        return [ # Return the ordered conversation messages
            SystemMessage(content=self.instruction), # Create the system instruction message
            HumanMessage(content=self.question), # Create the human question message
        ] # Finish the message list


prompt = InstructionPromptValue( # Create the concrete prompt value
    instruction="Answer clearly and concisely.", # Provide the system instruction
    question="What is retrieval-augmented generation?", # Provide the user's question
) # Finish creating the prompt value

text_prompt = prompt.to_string() # Convert the prompt value into text
chat_messages = prompt.to_messages() # Convert the prompt value into chat messages

print(text_prompt) # Display the text-generation representation

for message in chat_messages: # Iterate through the chat-message representation
    print(type(message).__name__, ":", message.content) # Display each message type and content

print(prompt.is_lc_serializable()) # Check whether prompt values support LangChain serialization
print(prompt.get_lc_namespace()) # Display the inherited serialization namespace


# `StringPromptValue: PromptValue`

Represents a text prompt.

## Fields

```python
text: str # Prompt text
type: Literal["StringPromptValue"] = "StringPromptValue" # Serialized prompt-value type
```

## Constructor

```python
StringPromptValue(
    *,
    text: str, # Prompt text
    type: Literal["StringPromptValue"] = "StringPromptValue", # Serialized prompt-value type
) -> None
```

## Methods

### `get_lc_namespace`

Returns the string-prompt serialization namespace.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # ["langchain", "prompts", "base"]
```

### `to_string`

Returns `text` unchanged.

```python
to_string(
    self,
) -> str # Prompt text
```

### `to_messages`

Wraps `text` in a `HumanMessage`.

```python
to_messages(
    self,
) -> list[BaseMessage] # One HumanMessage containing the prompt text
```

In [ ]:
from langchain_core.prompt_values import StringPromptValue # Import the text-based prompt value class


prompt_value = StringPromptValue( # Create a StringPromptValue object
    text="Explain machine learning in simple words.", # Store the text prompt
) # Finish creating the prompt value

text_prompt = prompt_value.to_string() # Convert the prompt value into a plain string
message_list = prompt_value.to_messages() # Convert the prompt value into chat messages

print("Text:", text_prompt) # Display the unchanged text prompt
print("Type:", prompt_value.type) # Display the serialized prompt-value type
print("Namespace:", prompt_value.get_lc_namespace()) # Display the LangChain serialization namespace

for message in message_list: # Iterate through the generated message list
    print("Message class:", type(message).__name__) # Display the generated message class
    print("Message content:", message.content) # Display the HumanMessage content

# `ChatPromptValue: PromptValue`

Represents a prompt built from chat messages.

## Fields

```python
messages: Sequence[BaseMessage] # Prompt messages
```

## Constructor

```python
ChatPromptValue(
    *,
    messages: Sequence[BaseMessage], # Prompt messages
) -> None
```

## Methods

### `to_string`

Converts the message sequence using `get_buffer_string`.

```python
to_string(
    self,
) -> str # Buffered string representation
```

### `to_messages`

Returns a new list containing the configured messages.

```python
to_messages(
    self,
) -> list[BaseMessage] # Copied message list
```

### `get_lc_namespace`

Returns the chat-prompt serialization namespace.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # ["langchain", "prompts", "chat"]
```

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage # Import LangChain chat message classes
from langchain_core.prompt_values import ChatPromptValue # Import the chat-based prompt value class


prompt_value = ChatPromptValue( # Create a prompt from an ordered sequence of messages
    messages=[ # Provide the messages that form the chat prompt
        SystemMessage(content="Answer as a helpful Python tutor."), # Add the system instruction
        HumanMessage(content="What is a Python list?"), # Add the user's question
        AIMessage(content="A list is an ordered and mutable collection."), # Add an earlier AI response
        HumanMessage(content="Show me a small example."), # Add the user's follow-up question
    ] # Finish the message sequence
) # Finish creating the ChatPromptValue object

text_prompt = prompt_value.to_string() # Convert all messages into one buffered string
copied_messages = prompt_value.to_messages() # Get a new list containing the configured messages
namespace = prompt_value.get_lc_namespace() # Get the LangChain serialization namespace

print("String representation:") # Display a heading for the converted prompt
print(text_prompt) # Display the buffered string representation

print("\nMessage representation:") # Display a heading for the message list
for message in copied_messages: # Process each returned message
    print(f"{type(message).__name__}: {message.content}") # Display the message type and content

print("\nNamespace:", namespace) # Display the serialization namespace
print("New list returned:", copied_messages is not prompt_value.messages) # Verify that to_messages returns a new list

# `ImageURL: TypedDict`

OpenAI-format image URL data used by multimodal prompt templates.

Declared with `total=False`; both fields are optional.

```python
detail: Literal["auto", "low", "high"] # Requested image-detail level
url: str # Image URL or base64-encoded image data
```

The `detail` value is not validated locally. Invalid values are left for the downstream API to reject.

In [ ]:
from langchain_core.prompt_values import ImageURL # Import the ImageURL TypedDict


def display_image_data(image: ImageURL) -> None: # Define a function that accepts ImageURL data
    print("URL:", image.get("url", "Not provided")) # Read the optional image URL safely
    print("Detail:", image.get("detail", "Not provided")) # Read the optional detail level safely


remote_image: ImageURL = { # Create image data using a remote URL
    "url": "https://example.com/product-image.jpg", # Provide the image URL
    "detail": "high", # Request high-detail image processing
} # Finish the remote image dictionary

base64_image: ImageURL = { # Create image data using base64-encoded content
    "url": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...", # Provide base64 image data
    "detail": "low", # Request lower-detail image processing
} # Finish the base64 image dictionary

url_only_image: ImageURL = { # Create image data without the optional detail key
    "url": "https://example.com/chart.png", # Provide only the image URL
} # Finish the URL-only dictionary

display_image_data(remote_image) # Display the remote image information
display_image_data(base64_image) # Display the base64 image information
display_image_data(url_only_image) # Display the image with no detail setting

# `ImagePromptValue: PromptValue`

Represents an image prompt using OpenAI-format image URL data.

## Fields

```python
image_url: ImageURL # Image URL data
type: Literal["ImagePromptValue"] = "ImagePromptValue" # Serialized prompt-value type
```

## Constructor

```python
ImagePromptValue(
    *,
    image_url: ImageURL, # Image URL data
    type: Literal["ImagePromptValue"] = "ImagePromptValue", # Serialized prompt-value type
) -> None
```

## Methods

### `to_string`

Returns the `url` value, or an empty string when the key is absent.

```python
to_string(
    self,
) -> str # Image URL or an empty string
```

### `to_messages`

Creates one `HumanMessage` whose content contains the image URL dictionary.

```python
to_messages(
    self,
) -> list[BaseMessage] # One multimodal HumanMessage
```

In [ ]:
from langchain_core.prompt_values import ImagePromptValue # Import the image-based prompt value class


prompt_value = ImagePromptValue( # Create an image prompt value
    image_url={ # Provide the OpenAI-format image data
        "url": "https://example.com/product-image.jpg", # Specify the image URL
        "detail": "high", # Request high-detail image processing
    } # Finish the image URL dictionary
) # Finish creating the ImagePromptValue object

image_url = prompt_value.to_string() # Extract the URL as a plain string
messages = prompt_value.to_messages() # Convert the image prompt into chat messages

print("Type:", prompt_value.type) # Display the serialized prompt-value type
print("Image URL:", image_url) # Display the URL returned by to_string
print("Message class:", type(messages[0]).__name__) # Display the generated message class
print("Message content:", messages[0].content) # Display the multimodal HumanMessage content

# `ChatPromptValueConcrete: ChatPromptValue`

Chat prompt value that narrows `messages` to the explicit `AnyMessage` union for external schemas.

## Fields

```python
messages: Sequence[AnyMessage] # Explicitly typed prompt messages
type: Literal["ChatPromptValueConcrete"] = "ChatPromptValueConcrete" # Serialized prompt-value type
```

## Constructor

```python
ChatPromptValueConcrete(
    *,
    messages: Sequence[AnyMessage], # Explicitly typed prompt messages
    type: Literal["ChatPromptValueConcrete"] = "ChatPromptValueConcrete", # Serialized prompt-value type
) -> None
```

It inherits string conversion, message-list conversion, and the chat-prompt serialization namespace from `ChatPromptValue`.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage # Import concrete LangChain message classes
from langchain_core.prompt_values import ChatPromptValueConcrete # Import the concrete chat prompt value class


prompt_value = ChatPromptValueConcrete( # Create a concrete chat prompt value
    messages=[ # Provide an explicitly typed sequence of LangChain messages
        SystemMessage(content="You are a concise Python tutor."), # Add the system instruction
        HumanMessage(content="What is a dictionary in Python?"), # Add the user's question
        AIMessage(content="A dictionary stores key-value pairs."), # Add an earlier AI response
        HumanMessage(content="Give me a small example."), # Add the user's follow-up request
    ] # Finish the message sequence
) # Finish creating the prompt value

text_prompt = prompt_value.to_string() # Convert all messages into one buffered string
message_list = prompt_value.to_messages() # Return a new list containing the configured messages
namespace = prompt_value.get_lc_namespace() # Get the inherited chat-prompt serialization namespace

print("Type:", prompt_value.type) # Display the serialized prompt-value type
print("Namespace:", namespace) # Display the inherited serialization namespace

print("\nString representation:") # Display a heading for the text representation
print(text_prompt) # Display the messages converted into one string

print("\nMessage representation:") # Display a heading for the message representation
for message in message_list: # Iterate through the copied message list
    print(f"{type(message).__name__}: {message.content}") # Display each message class and content

print("\nNew list returned:", message_list is not prompt_value.messages) # Verify that to_messages returns a new list
print("Same message objects:", message_list[0] is prompt_value.messages[0]) # Verify that the message objects are reused